# Notebook 5 · Quantum-Inspired Tensor Networks

### SDS2026 Workshop — *Quantum AI: From Inspiration to Enhancement - A Practical Journey*

This lab studies tensor-train compression as a quantum-inspired method for
large weight matrices. No quantum hardware is involved.

Key terms:

- **LLM** (Large Language Model): the model class motivating large weight-matrix compression.
- **MPS** (Matrix Product State): a chain-structured tensor-network format.
- **TT** (Tensor Train): the same factorization language used in numerical linear algebra and ML compression.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pennylane as qml
import pennylane.numpy as pnp   # PennyLane-aware numpy – needed for autograd
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')
print('PennyLane', qml.__version__)

## How to read this notebook

The examples below all follow the same four-step pattern:

1. **Encode a business question as variables.** For optimisation this usually means a bitstring, such as `10100`.
2. **Turn the rules into a cost.** Good answers receive low energy; illegal answers receive penalties.
3. **Let an algorithm shape a probability distribution.** QAOA, VQE, kernels, and tensor methods do this in different ways.
4. **Read out and verify.** A quantum device returns measured outcomes; in most near-term workflows those outcomes are checked classically.

The important point is not that a bitstring is magical. The point is that many useful decision problems can be encoded as bitstrings, and quantum circuits naturally return bitstrings after measurement. The research question is whether the quantum state evolution before measurement can bias those samples toward better candidates faster, or with less memory, than a classical method.

In [ ]:
# ── Mental model: from business question to measured bitstrings ─────────────
from matplotlib.patches import FancyBboxPatch

fig, ax = plt.subplots(figsize=(11, 2.8))
ax.axis("off")

boxes = [
    ("Business question", "choose assets\nroute cities\nassign labels"),
    ("Bitstring encoding", "10100\ncity-position bits"),
    ("Cost / Hamiltonian", "low energy = good\npenalties = illegal"),
    ("Quantum / hybrid loop", "prepare state\nmix + phase\noptimise angles"),
    ("Readout + check", "sample bitstrings\nverify feasibility"),
]

for i, (title, body) in enumerate(boxes):
    x = 0.02 + i * 0.195
    rect = FancyBboxPatch(
        (x, 0.28),
        0.16,
        0.45,
        boxstyle="round,pad=0.02",
        linewidth=1.4,
        edgecolor="#334155",
        facecolor="#eef2ff",
        transform=ax.transAxes,
    )
    ax.add_patch(rect)
    ax.text(x + 0.08, 0.61, title, ha="center", va="center",
            fontsize=9.5, fontweight="bold", transform=ax.transAxes)
    ax.text(x + 0.08, 0.43, body, ha="center", va="center",
            fontsize=8.5, transform=ax.transAxes)
    if i < len(boxes) - 1:
        ax.annotate(
            "",
            xy=(x + 0.185, 0.505),
            xytext=(x + 0.165, 0.505),
            arrowprops={"arrowstyle": "->", "lw": 1.4, "color": "#334155"},
            xycoords=ax.transAxes,
        )

ax.text(
    0.5,
    0.08,
    "Near-term value comes from the whole loop: encoding, distribution shaping, and classical verification.",
    ha="center",
    fontsize=9,
    transform=ax.transAxes,
)
plt.tight_layout()
plt.savefig("bitstring_pipeline.png", dpi=100)
plt.show()

## Combinatorial problems, NP, and bitstrings

A useful first mental model is: **many hard decision and optimisation problems can be written as “find a bitstring that satisfies rules, and among feasible bitstrings choose the best one.”**

This is close to the practical story, but it needs two caveats.

First, **NP** is formally a class of *decision* problems: yes/no questions whose proposed answers can be checked efficiently. In applications we often work with the related **search** or **optimisation** version: not just “does a valid solution exist?”, but “which assignment is valid and good?”. Many such problems are encoded as binary variables, QUBOs, or Ising Hamiltonians.

Second, a quantum computer does not output “the solution” automatically. After measurement it outputs a **bitstring sampled from a probability distribution**. The hope is that the quantum part prepares a distribution that puts more probability mass on useful bitstrings. Classical code still builds the encoding, tunes parameters, checks feasibility, and compares against baselines.

So the honest business pipeline is:

$$
	ext{problem} 
ightarrow 	ext{bitstring encoding} 
ightarrow
	ext{cost / Hamiltonian} 
ightarrow 	ext{quantum or hybrid search} 
ightarrow
	ext{classical verification}.
$$

When people say “huge matrix operations happen behind the scenes,” the precise statement is this: a quantum state over $n$ qubits has $2^n$ amplitudes. A classical simulator must store and update that exponentially large vector in general. Quantum hardware evolves the physical state directly and then returns samples. This does not guarantee speed-up for every task, but it explains why algorithms based on interference, entanglement, and measurement distributions are worth testing on certain structured problems.

---
## Section E · Workshop-led LLM-style compression (Tensor Train)

### E.0 — Canonical demo vs optional reading

The cells below are the **workshop track**: Tensor Train (TT) compression of a dense weight block and a small sklearn MLP stress test. Treat **CompactifAI** and similar product lines as *optional* follow-ups you explore after the mechanics are clear.

### E.1 — MPS / TT background

In many-body physics, **Matrix Product States (MPS)** compress wavefunctions; in numerical linear algebra the same factorisation is the **Tensor Train**. Bond dimension controls accuracy versus parameter count.

In [ ]:
import tensorly as tl
from tensorly.decomposition import tensor_train
print('TensorLy version:', tl.__version__)

### E.2 — Tensor Train compression of a weight matrix

We simulate the weight matrix of one MLP layer and compress it at varying bond dimensions.

In [ ]:
np.random.seed(123)

# ── Simulate a weight matrix (e.g. FFN layer in a small transformer) ──────
D_IN, D_OUT = 64, 64    # 64x64 = 4096 parameters
W = np.random.randn(D_IN, D_OUT).astype(np.float32)

# Reshape into a 4D tensor: (4, 16, 4, 16) — reshape to enable TT
SHAPE = (4, 16, 4, 16)
W_tensor = W.reshape(SHAPE)

original_params = W.size
print(f'Original weight matrix: {D_IN}×{D_OUT} = {original_params} parameters')
print(f'Reshaped tensor: {SHAPE}')

# ── Compress at different bond dimensions and measure error ───────────────
tl.set_backend('numpy')

bond_dims   = [1, 2, 4, 6, 8, 12]
errors      = []
param_counts= []

for r in bond_dims:
    # Decompose into TT with max rank r
    tt_cores = tensor_train(W_tensor, rank=[1, r, r, r, 1])

    # Reconstruct from cores
    W_rec = tl.tt_to_tensor(tt_cores).reshape(D_IN, D_OUT)

    # Relative Frobenius error
    err = np.linalg.norm(W - W_rec) / np.linalg.norm(W)
    errors.append(err)

    # Count compressed parameters
    n_compressed = sum(c.size for c in tt_cores)
    param_counts.append(n_compressed)

    ratio = original_params / n_compressed
    print(f'  rank={r:2d} | params={n_compressed:5d} | compression={ratio:.1f}× | error={err:.4f}')

In [ ]:
# ── Accuracy vs. Compression trade-off ───────────────────────────────────
compression_ratios = [original_params / p for p in param_counts]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(bond_dims, errors, 'o-', color='steelblue', lw=2)
axes[0].set_xlabel('Bond dimension (TT rank)')
axes[0].set_ylabel('Relative Frobenius error')
axes[0].set_title('Approximation Error vs. Bond Dimension')
axes[0].grid(alpha=0.3)

axes[1].plot(compression_ratios, errors, 's-', color='darkorange', lw=2)
axes[1].set_xlabel('Compression ratio (×)')
axes[1].set_ylabel('Relative Frobenius error')
axes[1].set_title('Error vs. Compression Ratio')
axes[1].invert_xaxis()   # high compression on right
axes[1].grid(alpha=0.3)

plt.suptitle('Tensor Train Compression of a Weight Matrix', fontsize=13)
plt.tight_layout()
plt.savefig('tt_compression.png', dpi=80)
plt.show()

### E.3 — Applying TT compression to a sklearn model (surrogate demo)

As a hands-on proxy for LLM compression, we:
1. Train a small MLP classifier on the digits dataset
2. Extract one hidden-layer weight matrix
3. Replace it with its TT approximation
4. Measure the accuracy impact

In [ ]:
from sklearn.datasets import load_digits
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# ── Train a small MLP ─────────────────────────────────────────────────────
digits = load_digits()
X_d, y_d = digits.data / 16.0, digits.target
X_tr, X_te, y_tr, y_te = train_test_split(X_d, y_d, test_size=0.3, random_state=0)

mlp = MLPClassifier(hidden_layer_sizes=(64,), max_iter=400, random_state=0)
mlp.fit(X_tr, y_tr)
acc_original = accuracy_score(y_te, mlp.predict(X_te))
print(f'Original MLP accuracy: {acc_original:.4f}')

# ── Compress the hidden→output weight matrix ──────────────────────────────
W_orig = mlp.coefs_[0].copy()   # shape (64, 64)
print(f'Weight matrix shape: {W_orig.shape}')

In [ ]:
results = []
for r in [1, 2, 4, 8, 16, 32]:
    # TT decomposition
    W_t  = W_orig.reshape(SHAPE)
    cores = tensor_train(W_t, rank=[1, r, r, r, 1])
    W_tt  = tl.tt_to_tensor(cores).reshape(D_IN, D_OUT)

    # Patch model
    mlp.coefs_[0] = W_tt
    acc = accuracy_score(y_te, mlp.predict(X_te))

    n_comp = sum(c.size for c in cores)
    ratio  = D_IN * D_OUT / n_comp
    results.append({'rank': r, 'ratio': ratio, 'accuracy': acc})
    print(f'  rank={r:2d} | {ratio:5.1f}× compression | accuracy={acc:.4f}')

mlp.coefs_[0] = W_orig   # restore original

# ── Plot ──────────────────────────────────────────────────────────────────
ratios = [r['ratio'] for r in results]
accs   = [r['accuracy'] for r in results]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(ratios, accs, 'D-', color='purple', lw=2, ms=8)
ax.axhline(acc_original, color='green', ls='--', lw=1.5, label=f'Uncompressed ({acc_original:.3f})')
ax.set_xlabel('Compression ratio (×)')
ax.set_ylabel('Test accuracy')
ax.set_title('MLP Accuracy under TT Weight Compression (Digits Dataset)')
ax.legend()
ax.grid(alpha=0.3)
ax.invert_xaxis()
plt.tight_layout()
plt.savefig('tt_mlp_accuracy.png', dpi=80)
plt.show()

### E.4 — Optional external directions (self-paced)

- **CompactifAI** (Multiverse Computing): TT-style compression of LLM weight blocks; see the paper for reported compression factors on LLaMA-class models.
- Compare against quantisation and pruning baselines before any production claim.

| Concept | Quantum physics | Machine learning |
|---------|-----------------|------------------|
| Structure | Matrix Product State (MPS) | Tensor Train (TT) |
| Free parameter | Bond dimension $\chi$ | TT rank $r$ |
| Governs | Entanglement / expressivity | Accuracy / compression |